# Multi-class Classification - Reuters Dataset

Dataset by Reuters in 1986 as a set of short newswires and their topics. (https://keras.io/api/datasets/reuters/)
Previously we looked at predicting a binary-classification problem; however, the same techniques can be used to predict many different classes. In the case of the Reuters dataset, we have 46 topics which can be predicted.

Much of what we're covering here are detailed in Chollet (2021) Section 3.4 Getting Started with Neural Networks: Classification and Regression - 4.2 Classifying newswires: A multiclass classification example, if you wish to also read along.

## Requirements
As previously, if we encounter a `ModuleNotFound` error, we should install the appropriate packages. You should already have all relevant packages from last week; however, you can uncomment the magic-commands below if you need to reinstall these.

In [ ]:
%pip install keras
%pip install tensorflow
%pip install numpy
%pip install scikit-learn
%pip install pandas

We can get rid of those pesky info / warning messages which tensorflow outputs and clogs up our output with the following. Make sure to do this prior to importing tensorflow.

In detail:

* 0 = all messages are logged (default behavior)
* 1 = INFO messages are not printed
* 2 = INFO and WARNING messages are not printed
* 3 = INFO, WARNING, and ERROR messages are not printed

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

## Introduction


In [ ]:
from keras.datasets import reuters

# Utility function provided by Keras.
# num_words limits the results to the most frequent 10K words in the dataset; these will be our number of features.
# test_split - The function already performs shuffled train-test split for us, we can define how big the test-split should be.
# seed - Random seed for shuffling. Delete if you want randomness everytime.

#reuters.load_data(num_words = 10000)
(X_train, y_train), (X_test, y_test) = reuters.load_data(num_words = 10000, test_split = 0.3, seed=42)

In [ ]:
print(f"X Train length: {len(X_train)}")
print(f"y Train length: {len(y_train)}")
print(f"X Test length: {len(X_test)}")
print(f"y Test length: {len(y_test)}")

print(f"Total data examples: {len(X_train) + len(X_test)}")

Let's investigate what the data looks like.

In [ ]:
print(X_train[0])

# And the label
print(y_train[0]) # -> 3

Our input data is simply a sequence of word indices. E.g `[1, 39, 556, 11]` is a sequence of word 1, 39, 556, and 11. This seems cryptic, but each index here links to a vocabulary entry. That is, a mapping from id -> word.

In [ ]:
vocabulary = reuters.get_word_index()
vocabulary = {v:k for k,v in vocabulary.items()} # Invert the mapping. Numeric indices as keys, words as values. Enables lookup.

for k,v in vocabulary.items():
    print(k, v)

For our first news article (index 0), let's go through each of those word indices and look up what the corresponding word from the article is. This should give us a better idea that the articles are just encoded as numbers linking to a fixed vocabulary. With the problem being numeric, we can now begin to solve *this* problem. 

Note: If we were to change the vocabulary mapping, then any trained models would no longer work and we'd need to re-train on the new mapping.

In [ ]:
for word_idx in X_train[0]:
    print( vocabulary.get(word_idx - 3, '?') ) # Indices 0, 1, and 2 are reserved. Therefore we offset by 3. If a word cannot be found, use a '?'.

## Pre-processing
At the moment our labels are integers. For Categorical Cross-entropy, we will require a column per outcome. E.g If we have 3 classes, then our y label should have 3 features. We call this `one-hot encoding`. Where the appropriate class is set to 1 (hot).

E.g For Class 3 we would have the vector `[0, 0, 1]`; Class 2 `[0, 1, 0]`; and Class 1 `[1, 0, 0]`. Vectors such as `[0, 0, 0]` and `[1, 1, 0]` or `[1, 1, 1]` would be erroneous.

E.g For a 5 class problem we would have a vector `[0, 0, 0, 0, 0]`, then just set the class = 1. E.g Class 3 would be `[0, 0, 1, 0, 0]`.

We will use this same concept to convert our input data - a 10,000 word vocabulary mapping - as well as our target/labels ready for our neural network to understand.

In [ ]:
import numpy as np

# First create an empty numpy matrix full of 0.
# Shape will be the number of rows of data we have, and at each row we will have a 10,000 column vector.
x_train_enc = np.zeros( shape=(len(X_train),10000) )
print(x_train_enc.shape)

# E.g 7859, 10000
# Each row has a 10K dimension vector stored. Each index of that 10K vector represent a word index.

for row_number, word_idx_seq in enumerate(X_train):
    # Wherever we have a word_index, set the appropriate index in the 10K vector to 1
    # E.g [ 1, 39, 566 ] would set index 1, 39, and 566 of our 10K vector to 1. All else would be 0.
    x_train_enc[ row_number, word_idx_seq] = 1

print(x_train_enc[0])


### Same for the x_test set now. Yes, we could make a function for this.


x_test_enc = np.zeros( shape=(len(X_test),10000) )
for row_number, word_idx_seq in enumerate(X_test):
    x_test_enc[ row_number, word_idx_seq] = 1

print(x_test_enc[0])


We can use a numpy utility to help us convert the labels. Going from integers -> vectors for each row.

In [ ]:
from keras.utils import to_categorical
y_train_enc = to_categorical(y_train)
y_test_enc = to_categorical(y_test)

In [ ]:
print(y_train_enc.shape) # 46 dimensional vector now.

## Model Definition
Now we have our input data in a format which works for us (one-hot encoding), let's define a model. Previously we defined quite a small Neural Network as our 'problem' was much simpler (wine). Today we're going to explore how changing these impacts our loss metrics, and how we can make some decisions on our architecture based on those.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

Model from last week:

```python
model = keras.Sequential()
model.add(layers.InputLayer(input_shape=(11,))) # 11 Columns of input
model.add(layers.Dense(32, activation="relu"))
model.add(layers.Dense(1, activation="sigmoid")) # 0->1 floating
model.summary()
```

A few changes we will make:
* Input shape changing from 11-dimension to 1000-dimension
* Additional hidden layer
* Output number of neurons will be 46 - to represent the number of news topics.
* Change of activation in final layer to `softmax`. Softmax outputs a probability distribution. We want a N-dimensional vector output where we can just take the argmax to find the predicted class.

In [ ]:
model = keras.Sequential()
model.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(64, activation="relu")) # 2 Hidden Layers
model.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
model.summary()

Previously we specified `optimizer='sgd'`, if we want more control we should create our optimisers explicitly and define their values. This will become important in later weeks. For now we will define a standard SGD optimiser, with default learning rate.

```python
from keras.optimizers import SGD

model.compile(...
    optimizer=SGD(learning_rate=0.01)
...)
```

Additionally, as we are **NOT** doing a binary classification problem we must change our loss function to account for the categoric approach we've taken. Instead of **binary**_crossentropy we will be using **categorical**_crossentropy

```python
model.compile(
    loss='categorical_crossentropy',
    ...
)
```

In [ ]:
from tensorflow.keras.optimizers import SGD

# Compile the model.
model.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate
    metrics=['accuracy']
)

### Training

Here we fit our model using the standard procedure from last week. We stored the training history in an object for plotting later, providing metrics per epoch. We provide training x and y data, as well as a number of epochs.

A new change for this week, is the introduction of the `batch_size` keyword. This allows us to control the mini-batch which is used in the optimiser (SGD). Recall from last week's reading the range from entire-data to single example for calculation of gradients. Mini-batches are the middle ground. By default Keras uses batch sizes of 32. It should be noted that this parameter has diminishing returns. Using a batch size of 64 is not twice as good as 32. For more complex data, we run into memory limitations where we cannot physically store that many data examples in one-go.

Additionally, we're going to provide a `validation set` for it to calculate validation loss automaticaly on. We're going to grab 1000 examples from our training set to hold back for this purpose. This is important for us to make quick decisions from the training and validation set differences as to how to tweak our model.

In [ ]:
# Slice notation
x_val = x_train_enc[:1000] # Grab from 0 -> 1000
y_val = y_train_enc[:1000]

x_train_enc_rest = x_train_enc[1000:] # From 1000 -> end
y_train_enc_rest = y_train_enc[1000:]

model_training_history = model.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 120,
    validation_data = (x_val, y_val)
)

### Evaluating

Let's use the plotting functionality covered last week. We will modify it to overlay the validation set loss over the training set on a single figure. Here we have gone back in and used `plt.axvline()` function to plot a vertical line.

In [ ]:
import matplotlib.pyplot as plt

# If we want to be fancy, we can set a theme by uncommenting the below line.
#plt.style.use('ggplot')

# No subplots here, as we're just drawing over the same plot with multiple data.

# Loss and Val Loss metrics.
loss = model_training_history.history['loss']
val_loss = model_training_history.history['val_loss']


# Training loss, and Validation loss. Providing labels is useful for our legend later.
plt.plot(loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
    
# X and Y axes labels.
plt.ylabel('Loss')
plt.xlabel('epochs')

# Draw a red vertical line at x=60.
plt.axvline(x=60, color='red')

# Display a legend.
plt.legend()

Based on our results here we begin to overfit around the x=60 - x=100 mark. In the end we have a training accuracy of approx 93% with validation accuracy of 77%.

At this point we should re-train our whole model, stopping at Epoch = 60. Then we can evaluate our model.

##### Task: Reinitialise model, retrain cutting training off at the optimal point
**TODO**:
* Add code below to re-create, re-compile, and re-train the NN setting epochs = 40 (or another suitable point where we begin to overfit)
* Add code to generate evaluation metrics for the entire test set. Note, A confusion matrix here would be a 46x46 matrix... Standard accuracy here is fine.

If we predict using the final model after 120 epochs, we've already overfit at this point. We know our model will not generalise well, and we can expect poor test-set performance. The reason for retraining and stopping at the 'sweet spot' is that we end up with a 'good' model which hasn't yet overfit and isn't underfit. It's the best it can currently be (given the architecture so far).

In [ ]:
### TODO:
# Re-create, re-compile, re-train the NN model, setting epochs = 40.

### HERE

pred_y = model.predict(x_test_enc)

# Let's look at the first prediction output. To get a feeling for the shape, and values.
print(pred_y[0].shape)
print(pred_y[0])
print(np.sum(pred_y[0])) # Softmax output. Should be approx = 1.

print(np.argmax(pred_y[0])) # Which label is the highest probability

### HERE some more comparison code for pred_y against ground truth (y_test) - NOT our one-hot encoded one. Note how Argmax provides us with the integer representation?

### Information Bottleneck
Let's modify our model definition to decrease the number of neurons in the hidden layer. This will behave as a 'pinch' going from 10000 -> 4 -> 4 -> 46.
Fundamentally, we're going to observe an inability of our neural network to model the problem's complexity well enough.

In theory, forcing a small bottleneck can be advantagegous to push the network to learn dense and compact representations of problems. But too much bottlenecking (and too quickly) can cause loss of information between stages.

In [ ]:
model2 = keras.Sequential()
model2.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.
model2.add(layers.Dense(4, activation="relu"))
model2.add(layers.Dense(4, activation="relu")) # 2 Hidden Layers
model2.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
model2.summary()

model2.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate for demonstration.
    metrics=['accuracy']
)

# Using train, test, val datasets from before.
model2_training_history = model2.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 120,
    validation_data = (x_val, y_val)
)

plt.plot(model2_training_history.history['loss'], label="Training Loss")
plt.plot(model2_training_history.history['val_loss'], label="Validation Loss")
plt.ylabel('Loss')
plt.xlabel('epochs')
plt.legend()

An overall accuracy of approx 62%. Validation accuracy of approx 60%. The overall loss is far from 0 indicating more training to be done; however, the validation loss has begun to increase indicating overfitting. This is an excellent candidate for adding more neurons / hidden layers. Once we get the training accuracy up, we can also then look at regularisation to address the overfitting.

### Dropout
#### Dropout: alpha = 0.2

In our first example, we might want to try using Dropout to help reduce the gap between validation loss curves and training loss curves. Let's set that up.

We can add Dropout using `layers.Dropout()` layer in our Sequential model, remembering to provide a percentage dropout. Usually 0.2-0.5 are good values.

Here we're going to train for a bit longer, as we would expect Dropout to increase training times slightly.

In [ ]:
model3 = keras.Sequential()
model3.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.

model3.add(layers.Dense(64, activation="relu"))
model3.add(layers.Dropout(0.2))

model3.add(layers.Dense(64, activation="relu")) # 2 Hidden Layers
model3.add(layers.Dropout(0.2))

model3.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
model3.summary()

model3.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate for demonstration.
    metrics=['accuracy']
)

# Using train, test, val datasets from before.
model3_training_history = model3.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 240, # train for a bit longer to get a better picture.
    validation_data = (x_val, y_val)
)

#### Dropout: alpha = 0.5
Let's also try a higher value for Dropout, alpha = 0.5

In [ ]:
model4 = keras.Sequential()
model4.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.

model4.add(layers.Dense(64, activation="relu"))
model4.add(layers.Dropout(0.5))

model4.add(layers.Dense(64, activation="relu")) # 2 Hidden Layers
model4.add(layers.Dropout(0.5))

model4.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
model4.summary()

model4.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate for demonstration.
    metrics=['accuracy']
)

# Using train, test, val datasets from before.
model4_training_history = model4.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 240, # train for a bit longer to get a better picture.
    validation_data = (x_val, y_val)
)

#### Evaluative graphs
##### Loss
We're going to overlay our original networks Training loss, and Validation Loss on this graph to see how we compare in an absolute way. Remember, the closer the loss is to 0, theoretically the better our network.

To differentiate between Training loss and Validation Loss, we've used dashed lines for training, and solid for validation. Specifically here you should focus on how low the overall loss goes, the divergence between architectures of their training and validation loss curves, and the moment of inflection of the validation loss curves.

In [ ]:
plt.plot(model3_training_history.history['loss'], '--', label="D0.2 - Train Loss")
plt.plot(model3_training_history.history['val_loss'], label="D0.2 - Val Loss")

plt.plot(model4_training_history.history['loss'], '--', label="D0.5 - Train Loss")
plt.plot(model4_training_history.history['val_loss'], label="D0.5 - Val Loss")

plt.plot(model_training_history.history['loss'], '--', label="Original Train loss")
plt.plot(model_training_history.history['val_loss'], label="Original Val loss")

plt.ylabel('Loss')
plt.xlabel('epochs')
plt.legend()

##### Accuracy

Looking at the accuracy graphs we can see that the original training accuracy curve improves faster than the others. Intuitively this makes sense, as we have introduced a form of reguluarisation, we may take longer to get to the 'end' of our training.

With Dropout, looking at our accuracy can also be misleading. During training time, we are dropping 20-50% of the neuron connections; however, at test time we are keeping them all.

In [ ]:
plt.plot(model3_training_history.history['accuracy'], '--', label="D0.2 - Train Acc")
plt.plot(model4_training_history.history['accuracy'], '--', label="D0.5 - Train Acc")
plt.plot(model_training_history.history['accuracy'], '--', label="Original Train Acc")

plt.ylabel('Acc')
plt.xlabel('epochs')
plt.legend()

#### Task: Plot Testing Accuracy

Using the three previously trained models, plot the testing accuracy on a chart. Does this fall in line with our expectations? How has the regularisation helped?

### Batch Normalisation

In our reading we introduced Batch Normalisation as a means to ensure that the variances from earlier layers do not drift too far as we propagate through our network. It also benefits us during training by reducing covariance shift; ensuring that gradient application doesn't then impact other layers too drastically.

For the application of this layer, we must separate out our non-linearities. We insert the BN between the activation function, and the neurons. Dense -> BN -> Activation. As per the original paper; however, recent code created by these authors shows BN applied after activation (ReLU) - this is still wildly debated in practice.

In [ ]:
bn_model = keras.Sequential()
bn_model.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.

bn_model.add(layers.Dense(64))
bn_model.add(layers.BatchNormalization())
bn_model.add(layers.ReLU())

bn_model.add(layers.Dense(64))# 2 Hidden Layers
bn_model.add(layers.BatchNormalization())
bn_model.add(layers.ReLU())

bn_model.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
bn_model.summary()

bn_model.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate for demonstration.
    metrics=['accuracy']
)

# Using train, test, val datasets from before.
bn_model_training_history = bn_model.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 150, # train for a bit longer to get a better picture.
    validation_data = (x_val, y_val)
)

In [ ]:
plt.plot(bn_model_training_history.history['loss'], '--', label="BN Train Loss")
plt.plot(bn_model_training_history.history['val_loss'], label="BN Val Loss")

plt.ylabel('Loss')
plt.xlabel('epochs')
plt.legend()

#### Obsevations
This training graph appears to improve training loss incredibly quickly, at a rate higher than most we've seen so far. However, alongside this, we notice that the validation loss has diverged significantly since around epoch 20, and is slowly going back up.

We would need to investigate the train / val splits to see if it's a data partitioning issue, then look at ways to add additional regularisation, or perhaps tweak model complexity.

### L1 / L2 Regularisation

In our reading we introduced a way to prevent weights from changing too much, or too drastically. Primarily we can look at doing this via L1, L2, or a mix of the two (L1_L2). This can be done for the weights, as well as the biases of a layer (and even to the activation output!).

Let's look at if this can help us with the above.

In [ ]:
bn_l2_model = keras.Sequential()
bn_l2_model.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.

bn_l2_model.add(layers.Dense(64, kernel_regularizer='l2'))
bn_l2_model.add(layers.BatchNormalization())
bn_l2_model.add(layers.ReLU())

bn_l2_model.add(layers.Dense(64, kernel_regularizer='l2'))# 2 Hidden Layers
bn_l2_model.add(layers.BatchNormalization())
bn_l2_model.add(layers.ReLU())

bn_l2_model.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
bn_l2_model.summary()

bn_l2_model.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate for demonstration.
    metrics=['accuracy']
)

# Using train, test, val datasets from before.
bn_l2_model_training_history = bn_l2_model.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 150, # train for a bit longer to get a better picture.
    validation_data = (x_val, y_val)
)

In [ ]:
plt.plot(bn_model_training_history.history['loss'], '--', label="BN Train Loss")
plt.plot(bn_model_training_history.history['val_loss'], label="BN Val Loss")

plt.plot(bn_l2_model_training_history.history['loss'], '--', label="BN L2 Train Loss")
plt.plot(bn_l2_model_training_history.history['val_loss'], label="BN L2 Val Loss")

plt.ylabel('Loss')
plt.xlabel('epochs')
plt.legend()

#### Observations

The L2 regularised approach seems to take longer to get to the plataeu, but generally results in the validation loss following the shape of the training loss. The only difference now is the large gap between them, which could be explained by dataset split differences. Our training accuracy is high, at approx 97%, and the validation loss appears to just about be flattening. We may want to look at training for more epochs to confirm a good stopping point.

### Final Remarks

![Training Curve](https://www.baeldung.com/wp-content/uploads/sites/4/2020/07/fitgraph.jpg)

Source: https://www.baeldung.com/cs/learning-curve-ml

#### Training Loss Curve not decreasing, or Training loss stagnates
This could indicate your model does not have sufficient capacity

#### Train and Validation Loss Diverging

If your validation loss diverges from your training loss, with your validation loss flatlining or even going up again then your model is overfit. Potential solutions:
* Add regularisation (Dropout / BN / L1 / L2 / etc)
* Reduce Model Capacity (if model is overly complex)
* Verify your Train and Validation splits are staistically comparable (Try change the shuffle seed.). Divergence could be caused by the sets being different.